# Phase 6: Backtest, Dual Bound & P&L Attribution

This notebook validates LSMC optimality via the dual bound, simulates battery degradation,
runs a synthetic 30-day backtest, and decomposes P&L via the attribution engine.

**Reference**: CLAUDE.md sections: Daily P&L attribution, Degradation model


In [ ]:
import sys, json, os, warnings
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config import ASSET, FINANCE, DEGRADATION, LSMC, VALIDATION
from src.asset.battery import BatteryAsset, initial_state
from src.asset.degradation import DegradationModel
from src.attribution.pnl_explain import (
    PnlExplainer, FactorChanges, backtest_summary, print_backtest_summary
)
print("Imports OK")


## 1. Battery Asset & Degradation

In [ ]:
battery   = BatteryAsset(ASSET)
deg_model = DegradationModel(DEGRADATION, ASSET)
print(battery.summary())

# Shadow price at SoH levels
for soh in [1.0, 0.95, 0.90, 0.85, 0.82]:
    lam = deg_model.shadow_price(soh)
    print(f"  SoH={soh:.2f} -> lambda_deg = GBP {lam:.2f}/MWh")


In [ ]:
# 15-year SoH trajectory
years_base, soh_base = deg_model.simulate_soh_trajectory(
    efc_per_year=520, life_years=15, avg_soc_frac=0.50,
    augment_years=ASSET["augment_years"])
years_hi,   soh_hi   = deg_model.simulate_soh_trajectory(
    efc_per_year=730, life_years=15, avg_soc_frac=0.55,
    augment_years=ASSET["augment_years"])
years_lo,   soh_lo   = deg_model.simulate_soh_trajectory(
    efc_per_year=400, life_years=15, avg_soc_frac=0.45,
    augment_years=ASSET["augment_years"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(years_base, soh_base*100, "b-",  lw=2,   label="Base (520 EFC/yr)")
ax.plot(years_hi,   soh_hi*100,   "r--", lw=1.5, label="High (730 EFC/yr, KYOS)")
ax.plot(years_lo,   soh_lo*100,   "g--", lw=1.5, label="Low  (400 EFC/yr)")
trig = ASSET["soh_augment_trigger"]
ax.axhline(trig*100, color="orange", lw=1.5, ls=":", label=f"Augment trigger ({trig:.0%})")
for yr in ASSET["augment_years"]: ax.axvline(yr, color="gray", lw=0.8, ls="--", alpha=0.6)
ax.set_xlabel("Year"); ax.set_ylabel("SoH (%)")
ax.set_title("SoH Trajectory — 200 MWh LFP BESS | Augmentations yr 4/8/12", fontweight="bold")
ax.legend(); ax.set_ylim(50, 105); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("soh_trajectory.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Dual Bound Verification

In [ ]:
POLICY_PATH = "../data/processed/lsmc_policy.pkl"
RESULT_PATH = "../data/processed/lsmc_valuation_result.pkl"
BUNDLE_PATH = "../data/processed/sim_bundle.pkl"

if all(os.path.exists(p) for p in [POLICY_PATH, RESULT_PATH, BUNDLE_PATH]):
    import pickle
    with open(POLICY_PATH, "rb") as f: policy = pickle.load(f)
    with open(RESULT_PATH, "rb") as f: val_result = pickle.load(f)
    with open(BUNDLE_PATH, "rb") as f: bundle = pickle.load(f)
    print(f"Loaded: {len(val_result.pv_paths):,} paths")
else:
    print("Phase-4 artefacts not found -- running mini LSMC")
    from src.processes.simulate import simulate, default_params_from_config
    from src.optimisation.lsmc import run_lsmc
    ss_p, hpfc_p, imb_p, anc_p = default_params_from_config()
    lsmc_dev = dict(LSMC); lsmc_dev["n_paths"]=100; lsmc_dev["n_steps"]=96
    bundle = simulate(ss_p, hpfc_p, imb_p, anc_p, n_paths=100, n_steps=96, dt=0.5/8760)
    policy, val_result = run_lsmc(bundle, ASSET, lsmc_dev, DEGRADATION, FINANCE)


In [ ]:
from src.optimisation.dual_bound import compute_dual_bound

dual_result = compute_dual_bound(
    bundle=bundle, policy=policy, val_result=val_result,
    asset_cfg=ASSET, lsmc_cfg=LSMC, deg_cfg=DEGRADATION, fin_cfg=FINANCE,
    n_dual_paths=100, threshold=LSMC["dual_gap_acceptable"], verbose=True,
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
labels = ["V_LSMC (lower)", "V_dual (upper)"]
vals   = [dual_result.v_lsmc, dual_result.v_dual]
colors = ["#3498db", "#e74c3c"]
bars = ax.barh(labels, vals, color=colors, edgecolor="white", height=0.45)
for bar in bars:
    w = bar.get_width()
    ax.text(w*1.01, bar.get_y()+bar.get_height()/2, f"GBP {w:,.0f}", va="center", fontsize=9)
gap_str = f"{dual_result.gap_pct:.2%}"
ok_str  = "PASS" if dual_result.dual_ok else "REFINE"
ax.set_xlabel("Value (GBP)")
ax.set_title(f"LSMC Optimality Gap: {gap_str} ({ok_str}) | target < {dual_result.threshold:.0%}",
             fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("dual_bound.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Synthetic 30-Day Backtest

In [ ]:
rng = np.random.default_rng(42)
N_DAYS = 30

mtm_base   = 50_000_000.0
daily_vol  = mtm_base * 0.008
mtm_series = np.zeros(N_DAYS + 1)
mtm_series[0] = mtm_base
for d in range(N_DAYS):
    mtm_series[d+1] = mtm_series[d] + rng.normal(0, daily_vol)

cf_expected = rng.normal(80_000, 15_000, N_DAYS)
cf_realised = cf_expected + rng.normal(0, 8_000, N_DAYS)

soh_model_daily  = np.linspace(1.000, 0.998, N_DAYS+1)
soh_actual_daily = np.clip(soh_model_daily + rng.normal(0, 0.0002, N_DAYS+1), 0.95, 1.0)

factor_changes = [
    FactorChanges(
        delta_baseload_gbp_mwh  = float(rng.normal(0, 1.5)),
        delta_vega_da_frac      = float(rng.normal(0, 0.02)),
        delta_dc_gbp_mwh        = float(rng.normal(0, 0.3)),
        delta_qr_gbp_mwh        = float(rng.normal(0, 0.2)),
        delta_imb_drift_gbp_mwh = float(rng.normal(0, 0.8)),
        delta_rho_bps           = float(rng.normal(0, 2.0)),
        delta_avail_pp          = float(rng.normal(0, 0.1)),
    ) for _ in range(N_DAYS)
]
print(f"Synthetic backtest: {N_DAYS} days ready")


In [ ]:
MTM_SUMMARY = "../data/processed/mtm_summary.json"
if os.path.exists(MTM_SUMMARY):
    with open(MTM_SUMMARY) as f:
        greeks_dict = json.load(f).get("greeks", {})
    print(f"Loaded {len(greeks_dict)} Greeks")
else:
    greeks_dict = {
        "delta_baseload": {"greek": 2_500_000.0, "bump_size": 1.0, "bump_unit": "GBP/MWh"},
        "delta_dc":       {"greek":   800_000.0, "bump_size": 1.0, "bump_unit": "GBP/MW/h"},
        "delta_qr":       {"greek":   400_000.0, "bump_size": 1.0, "bump_unit": "GBP/MW/h"},
        "vega_da":        {"greek": 3_000_000.0, "bump_size": 0.10,"bump_unit": "fraction"},
        "delta_imb_drift":{"greek":   600_000.0, "bump_size": 5.0, "bump_unit": "GBP/MWh"},
        "rho":            {"greek":-25_000_000.0,"bump_size":50.0, "bump_unit": "bps"},
        "delta_avail":    {"greek":-2_500_000.0, "bump_size":-2.0, "bump_unit": "pp"},
    }
    print("Using synthetic Greeks")

explainer = PnlExplainer(greeks_dict, ASSET, FINANCE)
results = explainer.explain_series(
    mtm_series=mtm_series, cf_realised=cf_realised, cf_expected=cf_expected,
    soh_actual=soh_actual_daily, soh_model=soh_model_daily,
    factor_changes=factor_changes,
)
results[0].print_waterfall()


In [ ]:
bk_summary = backtest_summary(results, VALIDATION)
print_backtest_summary(bk_summary)


## 4. Attribution Charts

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), height_ratios=[1.5, 1])

days   = np.arange(N_DAYS)
labels = ["Theta", "Delta-explain", "Exec surprise", "Deg surprise", "Residual"]
series = [
    np.array([r.theta for r in results]),
    np.array([r.total_delta_explain for r in results]),
    np.array([r.execution_surprise for r in results]),
    np.array([r.degradation_surprise for r in results]),
    np.array([r.residual for r in results]),
]
colors = ["#3498db", "#e74c3c", "#27ae60", "#f39c12", "#95a5a6"]

ax = axes[0]
bp = np.zeros(N_DAYS); bn = np.zeros(N_DAYS)
for s, lbl, col in zip(series, labels, colors):
    pos = np.where(s>0, s, 0); neg = np.where(s<0, s, 0)
    ax.bar(days, pos, bottom=bp, color=col, label=lbl, alpha=0.85)
    ax.bar(days, neg, bottom=bn, color=col, alpha=0.85)
    bp += pos; bn += neg
ax.axhline(0, color="#333", lw=0.8)
ax.set_ylabel("Daily ΔMTM (GBP)"); ax.set_title("Daily P&L Attribution", fontweight="bold")
ax.legend(loc="upper right", fontsize=8, ncol=3); ax.grid(axis="y", alpha=0.3)

ax2 = axes[1]
rpcts = [r.residual_pct*100 for r in results]
bcols = ["#e74c3c" if p>5 else "#27ae60" for p in rpcts]
ax2.bar(days, rpcts, color=bcols, edgecolor="white")
ax2.axhline(5, color="#e74c3c", lw=1.5, ls="--", label="5% target")
ax2.set_xlabel("Day"); ax2.set_ylabel("Residual (%)")
ax2.set_title("Daily Residual vs 5% Target", fontweight="bold")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("pnl_attribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Save Results

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

phase6 = {
    "dual_bound": {
        "v_lsmc":   dual_result.v_lsmc,
        "v_dual":   dual_result.v_dual,
        "gap_pct":  dual_result.gap_pct,
        "dual_ok":  dual_result.dual_ok,
    },
    "backtest": bk_summary,
    "soh_base_yr15": float(soh_base[-1]),
}

with open("../data/processed/phase6_summary.json", "w") as f:
    json.dump(phase6, f, indent=2, default=str)

print("Saved: data/processed/phase6_summary.json")
print(f"  Dual gap: {dual_result.gap_pct:.2%}")
gap_ok = "PASS" if dual_result.dual_ok else "REFINE"
print(f"  Status:   {gap_ok}")
print(f"  Backtest residual: {bk_summary.get(chr(116)+chr(111)+chr(116)+chr(97)+chr(108)+chr(95)+chr(114)+chr(101)+chr(115)+chr(105)+chr(100)+chr(117)+chr(97)+chr(108)+chr(95)+chr(112)+chr(99)+chr(116), 0):.2%}")
